In [22]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
# from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import warnings
warnings.filterwarnings("ignore")

from langchain_community.utilities.sql_database import SQLDatabase

sql_db = SQLDatabase.from_uri("sqlite:///SalesDB/sales.db")

llm = ChatOllama(
    model="mistral:latest",
    temperature=0
)
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=sql_db, llm=llm)
toolkit.get_tools()

# from langchain.agents import create_agent

# agent = create_agent(llm, toolkit.get_tools())
# agent

# example_query = "How much total sales we made for Tablet"

# events = agent.stream(
#     {"messages": [("user", example_query)]},
#     stream_mode="values",
# )
# for event in events:
#     event["messages"][-1].pretty_print()

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002D58EE2D9D0>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002D58EE2D9D0>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002D58EE2D9D0>),
 QuerySQLCheckerTool(description='Use this tool to 

#### **SQL Agents**

In [ ]:
from langchain_community.utilities.sql_database import SQLDatabase

sql_db = SQLDatabase.from_uri("sqlite:///SalesDB/sales.db")

llm = ChatOllama(
    model="mistral:latest",
    temperature=0
)
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=sql_db, llm=llm)
toolkit.get_tools()


[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002D58EE4B2F0>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002D58EE4B2F0>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002D58EE4B2F0>),
 QuerySQLCheckerTool(description='Use this tool to 

In [12]:
print("Available Tools:")
for tool in tools:
    print("-", tool.name)

Available Tools:
- sql_db_query
- sql_db_schema
- sql_db_list_tables
- sql_db_query_checker


In [13]:
# CONVERT TOOL LIST TO DICT

tool_map = {
    tool.name: tool
    for tool in tools
}

In [ ]:
# STEP 1 : GET TABLES
tables = tool_map["sql_db_list_tables"].invoke({})

print("\nTables:")
print(tables)

# STEP 2 : GET SCHEMA
schema = tool_map["sql_db_schema"].invoke(
    {"table_names": "orders"}
)

print("\nSchema:")
print(schema)



Tables:
orders

Schema:

CREATE TABLE orders (
	id INTEGER, 
	customer_name TEXT NOT NULL, 
	product_name TEXT NOT NULL, 
	quantity INTEGER NOT NULL, 
	price REAL NOT NULL, 
	total REAL NOT NULL, 
	PRIMARY KEY (id)
)

/*
3 rows from orders table:
id	customer_name	product_name	quantity	price	total
1	John Doe	Laptop	1	1000.0	1000.0
2	Jane Smith	Smartphone	2	500.0	1000.0
3	Bob Johnson	Tablet	3	200.0	600.0
*/


In [20]:
# USER QUESTION
question = "How much total sales we made for Tablet"

# STEP 3 : ASK LLM TO GENERATE SQL

sql_prompt = f"""
You are a SQLite expert.

Database Schema:

{schema}

Question:

{question}

Return ONLY valid SQLite query.
Do not explain anything.
"""

sql_query = llm.invoke(sql_prompt).content.strip()

print("\nGenerated SQL:")
print(sql_query)

# STEP 4 : EXECUTE SQL
result = tool_map["sql_db_query"].invoke(
    {"query": sql_query}
)

print("\nSQL Result:")
print(result)

# STEP 5 : FINAL ANSWER
final_prompt = f"""
Question:
{question}

SQL Query:
{sql_query}

SQL Result:
{result}

Provide a business-friendly answer.
"""

# answer = llm.invoke(final_prompt)

# print("\nFinal Answer:")
# print(answer.content)

print("\nFinal Answer:\n")

for chunk in llm.stream(final_prompt):
    if hasattr(chunk, "content"):
        print(chunk.content, end="", flush=True)


Generated SQL:
SELECT SUM(total) FROM orders WHERE product_name = 'Tablet';

SQL Result:
[(600.0,)]

Final Answer:

 Based on the SQL query and result you've provided, it appears that the total sales for Tablets amount to 600.00. This means that your company has generated revenue of six hundred dollars from selling Tablets during the time period covered by the orders in question.

In [17]:
print("\nFinal Answer:\n")

for chunk in llm.stream(final_prompt):
    if hasattr(chunk, "content"):
        print(chunk.content, end="", flush=True)


Final Answer:

 Based on the SQL query and result you've provided, it appears that the total sales for Tablets amount to 600.00. This means that your company has generated revenue of six hundred dollars from selling Tablets during the time period covered by the orders in question.